# Étape 2: Visualisation avec ACP (PCA)

**Objectif:** Réduire les embeddings de dimension 50 à 2 dimensions pour visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import pickle

# Style pour de meilleurs graphiques
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Charger les données du notebook précédent

In [ ]:
# Charger la matrice d'embeddings
embedding_matrix = np.load('embedding_matrix.npy')

# Charger les dictionnaires
with open('word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)

with open('idx2word.pkl', 'rb') as f:
    idx2word = pickle.load(f)

print(f"Matrice d'embeddings chargée: {embedding_matrix.shape}")
print(f"Vocabulaire: {len(word2idx)} mots")

## 2. Application de l'ACP (PCA)

In [ ]:
# Appliquer PCA pour réduire à 2 dimensions
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embedding_matrix)

print(f"Forme après PCA: {embeddings_2d.shape}")
print(f"\nVariance expliquée par composante:")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")
print(f"\nVariance totale expliquée: {pca.explained_variance_ratio_.sum():.2%}")

## 3. Visualisation des mots dans l'espace 2D

In [ ]:
def plot_words_pca(embeddings_2d, idx2word, figsize=(14, 10), fontsize=10):
    """
    Visualiser tous les mots dans l'espace PCA 2D
    """
    plt.figure(figsize=figsize)
    
    # Tracer les points
    plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                alpha=0.6, s=100, c='steelblue', edgecolors='navy')
    
    # Ajouter les labels pour chaque mot
    for idx, word in idx2word.items():
        x, y = embeddings_2d[idx]
        plt.annotate(word, (x, y), 
                    fontsize=fontsize,
                    alpha=0.8,
                    xytext=(5, 5),
                    textcoords='offset points')
    
    plt.xlabel('Première Composante Principale (PC1)', fontsize=12)
    plt.ylabel('Deuxième Composante Principale (PC2)', fontsize=12)
    plt.title('Visualisation des Word Embeddings avec PCA', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Visualiser tous les mots
plot_words_pca(embeddings_2d, idx2word)

## 4. Visualisation avec couleurs par groupes sémantiques

In [ ]:
# Définir des groupes sémantiques manuellement (optionnel)
semantic_groups = {
    'Géographie': ['maroc', 'rabat', 'afrique', 'nord', 'nord.'],
    'IA/Tech': ['intelligence', 'artificielle', 'machine', 'learning', 'python', 
                'programmation', 'réseaux', 'neurones', 'deep'],
    'Général': ['est', 'un', 'une', 'le', 'la', 'de', 'du', 'pour', 'en']
}

def plot_words_by_groups(embeddings_2d, idx2word, semantic_groups):
    """
    Visualiser avec des couleurs différentes par groupe sémantique
    """
    plt.figure(figsize=(14, 10))
    
    colors = ['red', 'blue', 'green', 'purple', 'orange']
    
    # Créer un mapping mot -> groupe
    word_to_group = {}
    for group_name, words in semantic_groups.items():
        for word in words:
            if word in word2idx:
                word_to_group[word] = group_name
    
    # Tracer chaque groupe avec une couleur différente
    for i, (group_name, words) in enumerate(semantic_groups.items()):
        group_indices = [word2idx[w] for w in words if w in word2idx]
        if group_indices:
            group_coords = embeddings_2d[group_indices]
            plt.scatter(group_coords[:, 0], group_coords[:, 1],
                       label=group_name, alpha=0.6, s=150, 
                       c=colors[i % len(colors)])
    
    # Tracer les mots non groupés
    ungrouped_indices = [idx for idx, word in idx2word.items() 
                         if word not in word_to_group]
    if ungrouped_indices:
        ungrouped_coords = embeddings_2d[ungrouped_indices]
        plt.scatter(ungrouped_coords[:, 0], ungrouped_coords[:, 1],
                   label='Autres', alpha=0.4, s=100, c='gray')
    
    # Ajouter les labels
    for idx, word in idx2word.items():
        x, y = embeddings_2d[idx]
        plt.annotate(word, (x, y), fontsize=9, alpha=0.7,
                    xytext=(3, 3), textcoords='offset points')
    
    plt.xlabel('PC1', fontsize=12)
    plt.ylabel('PC2', fontsize=12)
    plt.title('Visualisation PCA avec Groupes Sémantiques', fontsize=14, fontweight='bold')
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_words_by_groups(embeddings_2d, idx2word, semantic_groups)

## 5. Analyse de la variance expliquée

In [ ]:
# Tester différents nombres de composantes
pca_full = PCA()
pca_full.fit(embedding_matrix)

# Plot de la variance expliquée cumulée
plt.figure(figsize=(10, 6))
cumsum_variance = np.cumsum(pca_full.explained_variance_ratio_)
plt.plot(range(1, len(cumsum_variance) + 1), cumsum_variance, 'bo-')
plt.axhline(y=0.8, color='r', linestyle='--', label='80% variance')
plt.axhline(y=0.9, color='g', linestyle='--', label='90% variance')
plt.xlabel('Nombre de Composantes', fontsize=12)
plt.ylabel('Variance Expliquée Cumulée', fontsize=12)
plt.title('Variance Expliquée par Nombre de Composantes PCA', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Nombre de composantes pour 80% variance: {np.argmax(cumsum_variance >= 0.8) + 1}")
print(f"Nombre de composantes pour 90% variance: {np.argmax(cumsum_variance >= 0.9) + 1}")

## 6. Sauvegarder les résultats PCA

In [ ]:
# Sauvegarder les embeddings 2D pour le clustering
np.save('embeddings_2d.npy', embeddings_2d)

# Sauvegarder le modèle PCA
import pickle
with open('pca_model.pkl', 'wb') as f:
    pickle.dump(pca, f)

print("✅ Résultats PCA sauvegardés!")
print("\nFichiers créés:")
print("  - embeddings_2d.npy")
print("  - pca_model.pkl")

## ✅ Étape 2 Complète!

**Observations:**
- Les mots sémantiquement similaires devraient être proches dans l'espace 2D
- La variance expliquée indique la qualité de la réduction de dimensionnalité

**Prochaine étape:** Notebook 3 - Clustering K-means